# 02 — Unified training: multi-seed reruns, loss-vs-data ablations, best vs final checkpoint

**Paper sections fed:** III-B (balancing ablations), III-C (fine-tuning sensitivity), III-E (checkpoint
comparison), III-F (seeds), Tables 4–6, compute table (training time).

**What this notebook does.** Runs a queue of experiments with the *same* architectures, freezing, learning
rates, loss, augmentation and schedule as the original 24 notebooks, and for every run saves:
`metrics.json` (spec, timings, params, best epoch), `history.csv` (per-epoch curves) and `preds.csv`
(softmax probabilities on **val and test** for the **best-val-loss** *and* **final-epoch** checkpoints).
All statistics are computed afterwards in notebook 03 from `preds.csv` — no GPU needed there.

**Kaggle use.** Add the original dataset (and the balanced dataset for `balanced` runs). Choose the queue in
cell 2, then *Save Version → Save & Run All (background)*. The run skips anything already finished: to resume
in a new session, attach the previous version's output as an input dataset (it is found automatically).

**Conditions**
| condition | training data | loss |
|---|---|---|
| `imbalanced` | original train (6,326) | inverse-frequency weighted CE (as paper) |
| `imbalanced_unweighted` | original train | plain CE — *no correction* |
| `oversample` | original train + fixed-augmented minority copies to 3,875/class (15,500) | plain CE |
| `balanced` | the paper's balanced set (8,000) | plain CE (as paper) |
| `downsample` | balanced set **without** its augmented copies: original COVID/Normal/TB + the same 2,000 Pneumonia (4,451) | plain CE |

**Revision options** (per-run `opts`, appear in the run id): `lungcrop=True` — train/val/test images cropped to the lung
bounding box from notebook 07 (Q1); `placement='multi'` — attention inside the residual branch of every bottleneck of ResNet50
layer3 and layer4 (canonical SE-/CBAM-ResNet position) plus the pre-pool module (10 modules, Q2); `pretrain='xrv'` — DenseNet121 with CXR-domain weights from TorchXRayVision (Q3);
`val='grouped'` — a 15%, patient-grouped validation split carved from the training set replaces the 38-image folder (Q8).
`balanced − downsample` isolates the augmentation of minority classes; `downsample − imbalanced_unweighted` isolates
downsampling Pneumonia (Q7).

*Why no "balanced + weighted loss"?* With 2,000 images per class the inverse-frequency weights are all 0.25, and
`CrossEntropyLoss(weight=w, reduction='mean')` divides by the summed weights, so the loss and gradients are
**identical** to the unweighted balanced run. The two conditions that do separate loss from data are
`imbalanced_unweighted` (loss effect = imbalanced − imbalanced_unweighted) and `oversample`
(oversampling effect without discarding 1,875 Pneumonia images).

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SHARED CODE — identical copy in notebooks 02, 04 and 05.
# Re-implements the 24 original notebooks in one place. Architectures, freezing,
# learning rates, loss, augmentation and schedule match the originals exactly;
# deviations are explicit, named options (see ORIGINAL-PIPELINE NOTES below).
# ═════════════════════════════════════════════════════════════════════════════
import os, re, json, time, math, random, hashlib, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']  # folder names (dataset spells TB this way)
NUM_CLASSES = 4
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ── ORIGINAL-PIPELINE NOTES (verified by reading the 20 shared notebooks) ──────
# * ResNet50: layer4 fine-tuned @1e-4; attention + fc @1e-3.
# * DenseNet121: denseblock4 + norm5 @1e-4; ReLU -> [attn] -> GAP -> fc @1e-3.
# * MobileNetV2: features[18] (final 1x1 conv 320->1280) @1e-4; [attn] + fc @1e-3.
#   The ORIGINAL BASELINE kept torchvision's Dropout(0.2) before fc, while the
#   SE/CBAM variants used no dropout. Replicated by default (head_dropout=None);
#   pass opts={'head_dropout': 0.0} to remove the confound.
# * Custom CNN: all layers @1e-3 from scratch; head 512->256->ReLU->Dropout(0.5)->4.
# * "Frozen" layers have requires_grad=False but stay in train() mode, so their
#   BatchNorm running statistics still update during training (replicated).
# * SE/CBAM hidden width = max(C//16, 4). Linear vs 1x1-conv MLP forms used in
#   different originals are mathematically equivalent.

# ─────────────────────────────── attention blocks ─────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.shared_mlp = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                        nn.Linear(hidden, channels, bias=False))
        self.avg_pool, self.max_pool = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
    def forward(self, x):
        b, c, _, _ = x.size()
        s = self.shared_mlp(self.avg_pool(x).view(b, c)) + self.shared_mlp(self.max_pool(x).view(b, c))
        return x * torch.sigmoid(s).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        m = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)
        return x * torch.sigmoid(self.conv(m))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

def make_attention(kind, channels):
    if kind == 'none': return nn.Identity()
    if kind == 'se':   return SEBlock(channels)
    if kind == 'cbam': return CBAM(channels)
    raise ValueError(kind)

# ─────────────────────────────── unified network ──────────────────────────────
class XRVInput(nn.Module):
    """For torchxrayvision (CXR-pretrained) weights: undo the ImageNet normalisation of channel 0
    and map [0, 1] intensities to the [-1024, 1024] single-channel range those weights expect.
    Keeps the data pipeline identical for every model."""
    def forward(self, x):
        return (x[:, :1] * STD[0] + MEAN[0]) * 2048.0 - 1024.0

class ReLUWrap(nn.Module):           # DenseNet's functional ReLU after features
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return F.relu(self.m(x))

def custom_cnn_features():
    layers, cin = [], 3
    for cout in [32, 64, 128, 256, 512]:
        layers += [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)]
        cin = cout
    return nn.Sequential(*layers)

class CXRNet(nn.Module):
    """body -> attn -> GAP -> head.  `attn` output (== input to GAP) is the
    uniform Grad-CAM target for all 12 configurations."""
    def __init__(self, arch, attention='none', pretrained=True, unfreeze='last', head_dropout=None,
                 pretrain='imagenet', placement='single'):
        super().__init__()
        self.arch, self.attention, self.placement = arch, attention, placement
        extra = []                                   # attention modules besides the pre-pool one
        if placement not in ('single', 'multi'): raise ValueError(placement)
        if placement == 'multi' and (arch != 'resnet50' or attention == 'none'):
            raise ValueError('multi-depth placement is implemented for ResNet50 with SE/CBAM only')
        if arch == 'resnet50':
            b = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            l3, l4 = b.layer3, b.layer4
            if placement == 'multi':                 # canonical SE-/CBAM-ResNet position: inside the residual branch
                for blk in list(b.layer3) + list(b.layer4):          # of every bottleneck of layer3 (6) and layer4 (3),
                    m = make_attention(attention, blk.bn3.num_features)  # after bn3, before the identity addition,
                    blk.bn3 = nn.Sequential(blk.bn3, m); extra.append(m)  # so the pretrained identity path is untouched;
                                                                         # + the pre-pool self.attn = 10 modules
            self.body = nn.Sequential(nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool), b.layer1, b.layer2, l3, l4)
            C = 2048
            ft = {'last': [self.body[4]], 'last2': [self.body[3], self.body[4]]}[unfreeze]
        elif arch == 'densenet121':
            if pretrain == 'xrv':                    # CXR-domain pretraining (torchxrayvision, 1-channel input)
                try:
                    import torchxrayvision as xrv
                except ImportError:                  # not preinstalled on Kaggle (needs Internet on)
                    import subprocess, sys
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'])
                    import torchxrayvision as xrv
                b = xrv.models.DenseNet(weights='densenet121-res224-all' if pretrained else None)
                f = b.features; self.body = nn.Sequential(XRVInput(), ReLUWrap(f))
            elif pretrain == 'imagenet':
                b = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
                f = b.features; self.body = ReLUWrap(f)
            else:
                raise ValueError(pretrain)
            C = 1024
            ft = {'last': [f.denseblock4, f.norm5],
                  'last2': [f.denseblock3, f.transition3, f.denseblock4, f.norm5]}[unfreeze]
        elif arch == 'mobilenetv2':
            if pretrain != 'imagenet': raise ValueError('CXR pretraining implemented for DenseNet121 only')
            b = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = b.features; C = 1280
            ft = {'last': [b.features[18]], 'last2': [b.features[17], b.features[18]]}[unfreeze]
        elif arch == 'customcnn':
            self.body = custom_cnn_features(); C = 512; ft = None
        else:
            raise ValueError(arch)
        self.attn = make_attention(attention, C)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if arch == 'customcnn':
            self.head = nn.Sequential(nn.Linear(C, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, NUM_CLASSES))
        else:
            if head_dropout is None:   # replicate originals
                head_dropout = 0.2 if (arch == 'mobilenetv2' and attention == 'none') else 0.0
            self.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(C, NUM_CLASSES)) if head_dropout > 0 \
                        else nn.Sequential(nn.Linear(C, NUM_CLASSES))
        self.channels = C
        # freezing
        if ft is not None:
            for p in self.body.parameters(): p.requires_grad = False
            for m in ft:
                for p in m.parameters(): p.requires_grad = True
        self._extra_attn = extra                     # plain list: not registered twice
        for m in extra:
            for p in m.parameters(): p.requires_grad = True
        self._ft_modules = ft

    def attention_parameters(self):
        ps = list(self.attn.parameters())
        for m in self._extra_attn: ps += list(m.parameters())
        return ps

    def forward(self, x):
        x = self.attn(self.body(x))
        return self.head(torch.flatten(self.pool(x), 1))

    def param_groups(self, lr_ft=1e-4, lr_new=1e-3, lr_attn=None, lr_scratch=1e-3):
        if self.arch == 'customcnn':
            return [{'params': list(self.parameters()), 'lr': lr_scratch, 'name': 'all'}]
        lr_attn = lr_new if lr_attn is None else lr_attn
        att = self.attention_parameters(); att_ids = {id(p) for p in att}
        g = [{'params': [p for m in self._ft_modules for p in m.parameters() if id(p) not in att_ids],
              'lr': lr_ft, 'name': 'finetuned_block'}]
        if self.attention != 'none':
            g.append({'params': att, 'lr': lr_attn, 'name': 'attention'})
        g.append({'params': list(self.head.parameters()), 'lr': lr_new, 'name': 'head'})
        return g

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    att = sum(p.numel() for p in model.attention_parameters())
    return {'params_total': tot, 'params_trainable': tr, 'params_attention': att}

# ─────────────────────────────── data ─────────────────────────────────────────
def list_split(root, split):
    items = []
    for ci, c in enumerate(CLASS_NAMES):
        d = os.path.join(root, split, c)
        if not os.path.isdir(d): raise FileNotFoundError(d)
        for fn in sorted(os.listdir(d)):               # same order as ImageFolder
            if fn.lower().endswith(IMG_EXTS):
                items.append((os.path.join(d, fn), ci))
    return items

def load_256(path, box=None):
    """Grayscale(3) -> Resize((256,256)) of the originals, cached as 1 channel.
    Converting back with .convert('RGB') reproduces the original tensors exactly
    (verified in the smoke test). `box` = (x0, y0, x1, y1) in [0, 1] crops the image first
    (lung-crop preprocessing; boxes from notebook 07)."""
    img = Image.open(path).convert('RGB').convert('L')   # == ImageFolder loader + Grayscale
    if box is not None:
        W, H = img.size; x0, y0, x1, y1 = box
        img = img.crop((int(round(x0 * W)), int(round(y0 * H)), int(round(x1 * W)), int(round(y1 * H))))
    img = img.resize((256, 256), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

_CACHE = {}
def cached_arrays(items, tag, verbose=True, boxes=None):
    key = (tag, len(items), items[0][0] if items else '')
    if key not in _CACHE:
        t = time.time()
        _CACHE[key] = np.stack([load_256(p, None if boxes is None else boxes.get(p)) for p, _ in items])
        if verbose: print(f'  cached {tag}: {len(items)} images in {time.time()-t:.0f}s')
    return _CACHE[key]

train_tf = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
# Paper Sec. III-B oversampling augmentation (applied once per synthetic copy, fixed per copy)
oversample_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.96, 1.04)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08)])

class CXRDataset(Dataset):
    """index entries: (array_row, label, copy_id). copy_id>0 => synthetic copy that
    receives a FIXED oversampling augmentation (like an offline-generated file)."""
    def __init__(self, arrays, labels, index, tf, seed=0):
        self.a, self.y, self.index, self.tf, self.seed = arrays, labels, index, tf, seed
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        r, y, copy_id = self.index[i]
        img = Image.fromarray(self.a[r]).convert('RGB')
        if copy_id > 0:
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed((self.seed * 1_000_003 + r * 1009 + copy_id) % (2**31))
                img = oversample_tf(img)
        return self.tf(img), y

def oversample_index(labels, seed):
    """Keep every original image; add fixed-augmented copies of minority classes
    until each class matches the majority count (no downsampling)."""
    labels = np.asarray(labels); rng = np.random.RandomState(seed)
    counts = np.bincount(labels, minlength=NUM_CLASSES); target = counts.max()
    idx = [(r, int(labels[r]), 0) for r in range(len(labels))]
    for c in range(NUM_CLASSES):
        rows = np.where(labels == c)[0]; need = target - len(rows); copy = 1
        while need > 0:
            order = rng.permutation(rows)
            for r in order[:need]:
                idx.append((int(r), c, copy))
            need -= min(need, len(order)); copy += 1
    return idx

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def _worker_init(wid):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

def patient_key(path):
    """Patient identifier recoverable from the filename (Kermany-style Pneumonia/Normal names);
    COVID-19 and Tuberculosis filenames carry none, so each image is its own group."""
    fn = os.path.basename(path)
    m = re.match(r'person(\d+)_', fn)
    if m: return 'P' + m.group(1)
    m = re.match(r'(NORMAL2-)?IM-(\d+)-', fn)
    if m: return ('N2-' if m.group(1) else 'N-') + m.group(2)
    return 'F:' + fn

def grouped_val_split(items, arrays, frac=0.15, seed=12345):
    """Hold out ~frac of each class as validation, keeping patients and pixel-identical duplicates together.
    The split is fixed (independent of the training seed). Returns (train_rows, val_rows)."""
    parent = {}
    def find(k):
        while parent.setdefault(k, k) != k: parent[k] = parent[parent[k]]; k = parent[k]
        return k
    keys = [patient_key(p) for p, _ in items]; first_with_hash = {}
    for r, k in enumerate(keys):
        h = hashlib.md5(arrays[r].tobytes()).hexdigest()
        if h in first_with_hash: parent[find(k)] = find(first_with_hash[h])
        else: first_with_hash[h] = k
    groups = {}
    for r, k in enumerate(keys): groups.setdefault(find(k), []).append(r)
    labels = np.array([y for _, y in items]); rng = np.random.RandomState(seed); val = set()
    for c in range(NUM_CLASSES):
        gs = sorted({find(keys[r]) for r in np.where(labels == c)[0]}); rng.shuffle(gs)
        target, n = frac * (labels == c).sum(), 0
        for g in gs:
            if n >= target: break
            rows = [r for r in groups[g] if r not in val]; val.update(groups[g]); n += sum(labels[r] == c for r in rows)
    val_rows = sorted(val); tr_rows = [r for r in range(len(items)) if r not in val]
    return tr_rows, val_rows

# ─────────────────────────────── metrics ──────────────────────────────────────
def confusion(y, p, k=NUM_CLASSES):
    return np.bincount(np.asarray(y) * k + np.asarray(p), minlength=k * k).reshape(k, k)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    spec = np.divide(tn, tn + fp, out=np.zeros_like(tp), where=(tn + fp) > 0)
    out = {'acc': tp.sum() / cm.sum(), 'macro_f1': f1.mean(), 'balanced_acc': rec.mean(),
           'weighted_f1': (f1 * cm.sum(1)).sum() / cm.sum()}
    for i, c in enumerate(CLASS_NAMES):
        out[f'prec_{c}'], out[f'rec_{c}'], out[f'f1_{c}'], out[f'spec_{c}'] = prec[i], rec[i], f1[i], spec[i]
    return out

def macro_f1_fast(y, p, k=NUM_CLASSES):
    return metrics_from_cm(confusion(y, p, k))['macro_f1']

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ORIGINAL_ROOT = '/kaggle/input/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis'
BALANCED_ROOT = '/kaggle/input/datasets/malihamostafa/balanceddatasetctbp'
OUT_DIR       = '/kaggle/working/results'
# Previous outputs (attached as inputs) are scanned so finished runs are skipped:
RESUME_GLOBS  = ['/kaggle/input/**/results/*/metrics.json', OUT_DIR + '/*/metrics.json']
LUNG_BOX_GLOBS = ['/kaggle/input/**/lungcrop/lung_boxes.csv', '/kaggle/working/lungcrop/lung_boxes.csv']  # from notebook 07

SEEDS        = [42, 0, 1, 2, 3]    # 5 seeds. Seeds 42/0/1 are already done and are skipped via the consolidated input.
BATCH_SIZE   = 32
NUM_EPOCHS   = 150
PATIENCE     = 20                  # early stopping on val loss (as paper)
LR_FT, LR_NEW, LR_SCRATCH = 1e-4, 1e-3, 1e-3
USE_AMP      = True                # fp16 autocast on T4 (~1.5–2x faster). Applied to EVERY rerun, so runs are
                                   # mutually consistent; set False for strict fp32 like the originals.
NUM_WORKERS  = 2
SAVE_WEIGHTS = 'best'              # 'best' | 'both' | 'none'  (Grad-CAM notebook needs 'best')
LOG_TEST_EVERY_EPOCH = False       # True = also log test macro-F1 each epoch (reporting only, never used for
                                   # selection) — direct evidence of how noisy 38-image val selection is. ~+8% time.
MAX_SESSION_HOURS = 11.3           # Kaggle limit is 12 h; stop launching new runs after this
SMOKE_TEST   = False               # True = tiny run for code checking

ALLOW_CPU    = False               # leave False: one ResNet50 run takes ~9 min on a T4 but ~8.6 h on Kaggle CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda' and not (ALLOW_CPU or SMOKE_TEST):
    raise RuntimeError('No GPU. In Kaggle: Settings → Accelerator → GPU T4 x2 (or P100), then run again. '
                       'On CPU a single run takes many hours and the queue cannot finish.')
os.makedirs(OUT_DIR, exist_ok=True)
print('device:', device, '| torch', torch.__version__)

ARCHS = ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']
ATTS  = ['none', 'se', 'cbam']
def spec(arch, att, cond, seed, **opts): return dict(arch=arch, attention=att, condition=cond, seed=seed, opts=opts)

# ── RUN QUEUE (priority order — highest value per GPU-hour first) ─────────────
TIER1 = [spec(a, t, 'imbalanced', s) for s in SEEDS for a in ARCHS for t in ATTS] + \
        [spec('resnet50', t, 'balanced', s) for s in SEEDS for t in ATTS]
TIER2 = [spec('resnet50', t, c, s) for s in SEEDS for c in ['imbalanced_unweighted', 'oversample'] for t in ['none', 'se']]
TIER3 = [spec(a, t, 'balanced', s) for s in SEEDS for a in ['densenet121', 'mobilenetv2', 'customcnn'] for t in ATTS]
SENS_SEEDS = [42, 0, 1]           # sensitivity runs use 3 seeds (as reported in the paper)
TIER4 = [spec('resnet50', 'se', 'imbalanced', s, attn_lr=1e-4) for s in SENS_SEEDS] + \
        [spec('resnet50', 'se', 'imbalanced', s, unfreeze='last2') for s in SENS_SEEDS] + \
        [spec('mobilenetv2', 'none', 'imbalanced', s, head_dropout=0.0) for s in SENS_SEEDS]
# TIER4 = sensitivity checks for Sec. III-C (attention LR = backbone LR; two blocks unfrozen) and the
#         MobileNetV2-baseline dropout confound.
# ── REVISION TIERS (reviewer questions) — all imbalanced/plain-CE unless stated, 5 seeds ──────────────
ATT3 = ['none', 'se', 'cbam']
REV1 = [spec('resnet50', t, 'downsample', s) for s in SEEDS for t in ['none', 'se']] + \
       [spec('resnet50', t, 'imbalanced', s, lungcrop=True) for s in SEEDS for t in ATT3]          # Q7, Q1
REV2 = [spec('resnet50', t, 'imbalanced', s, val='grouped') for s in SEEDS for t in ATT3] + \
       [spec('resnet50', t, 'imbalanced', s, placement='multi') for s in SEEDS for t in ['se', 'cbam']]  # Q8, Q2
REV3 = [spec('densenet121', t, 'imbalanced', s, pretrain='xrv') for s in SEEDS for t in ATT3]     # Q3
# REV1 ≈ 3.5 GPU-h, REV2 ≈ 3.5 GPU-h, REV3 ≈ 2.7 GPU-h on a T4 (measured per-run medians).
# Main-study runs (TIER1–4) are already finished and are skipped automatically if the consolidated output is attached.
QUEUE = TIER1 + TIER3 + TIER2 + TIER4 + REV1 + REV2 + REV3

In [ ]:
def run_id(s):
    o = s['opts']
    suf = ''.join(f'_{k}-{v}' for k, v in sorted(o.items()))
    return f"{s['arch']}_{s['attention']}_{s['condition']}_s{s['seed']}{suf}"

def finished_ids():
    done = set()
    for g in RESUME_GLOBS:
        for f in glob.glob(g, recursive=True):
            try:
                if json.load(open(f)).get('status') == 'done': done.add(os.path.basename(os.path.dirname(f)))
            except Exception: pass
    return done

_BOXES = None
def lung_boxes():
    """relpath -> (x0, y0, x1, y1) in [0, 1]; images failing mask QC get the full frame (counted in nb 07)."""
    global _BOXES
    if _BOXES is None:
        fs = [f for g in LUNG_BOX_GLOBS for f in glob.glob(g, recursive=True)]
        if not fs: raise FileNotFoundError('lung_boxes.csv not found — run notebook 07 and attach its output')
        B = pd.read_csv(fs[0])
        _BOXES = {os.path.join(ORIGINAL_ROOT, r.relpath): (r.x0, r.y0, r.x1, r.y1) for r in B.itertuples()}
        print(f'  lung boxes: {len(B)} images ({(~B.qc_ok).sum()} fall back to the full frame)')
    return _BOXES

_DOWN = None
def downsample_items(tr_bal):
    """Balanced-train images that are pixel-identical (after decoding/resizing) to an original training image,
    i.e. the balanced set minus its augmented copies — the same definition as the provenance audit (nb 01)."""
    global _DOWN
    if _DOWN is None:
        A_b = cached_arrays(tr_bal, 'train_bal_all', verbose=True)
        tr_o = list_split(ORIGINAL_ROOT, 'train')
        h_o = {hashlib.md5(a.tobytes()).hexdigest() for a in cached_arrays(tr_o, 'train_orig_all')}
        keep = [i for i, a in enumerate(A_b) if hashlib.md5(a.tobytes()).hexdigest() in h_o]
        _DOWN = [tr_bal[i] for i in keep]
        n_aug_name = sum(os.path.basename(p).lower().startswith('aug') for p, _ in _DOWN)
        print(f'  downsample: kept {len(_DOWN)} of {len(tr_bal)} balanced-train images as originals '
              f'(per class {np.bincount([y for _, y in _DOWN], minlength=NUM_CLASSES).tolist()}; '
              f'{n_aug_name} kept files have an aug* filename — should be 0)')
    return _DOWN

def build_data(cond, seed, opts=None):
    opts = opts or {}
    tr_root = BALANCED_ROOT if cond in ('balanced', 'downsample') else ORIGINAL_ROOT
    tr = list_split(tr_root, 'train')
    if cond == 'downsample':                 # balanced set WITHOUT its augmented copies: original minority images
        tr = downsample_items(tr)            # + the same 2,000 Pneumonia (identified by pixel identity, as in nb 01)
    va = list_split(ORIGINAL_ROOT, 'val')   # val/test ALWAYS from the original folders (audit nb 01 confirms
    te = list_split(ORIGINAL_ROOT, 'test')  # they are byte-identical to the balanced dataset's copies)
    if SMOKE_TEST:
        tr, va, te = tr[::max(1, len(tr) // 64)], va, te[::max(1, len(te) // 40)]
    boxes = lung_boxes() if opts.get('lungcrop') else None
    if boxes is not None and cond in ('balanced', 'downsample'):
        raise ValueError('lung-crop boxes exist for the original dataset only')
    sfx = '_lungcrop' if boxes is not None else ''
    A_tr = cached_arrays(tr, {'balanced': 'train_bal', 'downsample': 'train_down'}.get(cond, 'train_orig') + sfx, boxes=boxes)
    A_va, A_te = cached_arrays(va, 'val' + sfx, boxes=boxes), cached_arrays(te, 'test' + sfx, boxes=boxes)
    y_tr = np.array([y for _, y in tr])
    rows = list(range(len(tr)))
    if opts.get('val') == 'grouped':         # Q8: larger, patient-grouped validation split carved from train
        rows, vrows = grouped_val_split(tr, A_tr)
        va, A_va = [tr[r] for r in vrows], A_tr[vrows]
        print(f'  grouped validation: {len(vrows)} images held out of {len(tr)} '
              f'(per class {np.bincount(y_tr[vrows], minlength=NUM_CLASSES).tolist()})')
    if cond == 'oversample':
        index = oversample_index(y_tr, seed)
    else:
        index = [(r, int(y_tr[r]), 0) for r in rows]
    counts = np.bincount([y for _, y, _ in index], minlength=NUM_CLASSES)
    if cond == 'downsample': print('  downsample-only train counts:', counts.tolist())
    return (tr, A_tr, y_tr, index, counts), (va, A_va), (te, A_te)

def make_criterion(cond, counts):
    if cond == 'imbalanced':                     # exactly as original notebooks
        w = 1.0 / torch.tensor(counts, dtype=torch.float); w = w / w.sum()
        return nn.CrossEntropyLoss(weight=w.to(device)), w.tolist()
    return nn.CrossEntropyLoss(), None

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == 'cuda'):
            out = model(x.to(device, non_blocking=True))
        P.append(torch.softmax(out.float(), 1).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def eval_loss(model, loader, criterion):
    model.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == 'cuda'):
                out = model(x)
            tot += criterion(out.float(), y).item() * x.size(0); n += x.size(0)
    return tot / n

In [ ]:
def train_one(s):
    rid = run_id(s); rdir = os.path.join(OUT_DIR, rid); os.makedirs(rdir, exist_ok=True)
    seed_everything(s['seed']); o = s['opts']
    (tr, A_tr, y_tr, index, counts), (va, A_va), (te, A_te) = build_data(s['condition'], s['seed'], s['opts'])
    g = torch.Generator(); g.manual_seed(s['seed'])
    ld = lambda ds, sh: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh, num_workers=NUM_WORKERS, pin_memory=True,
                                   worker_init_fn=_worker_init, generator=g if sh else None, persistent_workers=NUM_WORKERS > 0)
    tr_loader = ld(CXRDataset(A_tr, y_tr, index, train_tf, s['seed']), True)
    va_loader = ld(CXRDataset(A_va, None, [(r, y, 0) for r, (_, y) in enumerate(va)], eval_tf), False)
    te_loader = ld(CXRDataset(A_te, None, [(r, y, 0) for r, (_, y) in enumerate(te)], eval_tf), False)

    model = CXRNet(s['arch'], s['attention'], pretrained=True, unfreeze=o.get('unfreeze', 'last'),
                   head_dropout=o.get('head_dropout', None), pretrain=o.get('pretrain', 'imagenet'),
                   placement=o.get('placement', 'single')).to(device)
    groups = model.param_groups(LR_FT, LR_NEW, o.get('attn_lr', None), LR_SCRATCH)
    optimizer = torch.optim.Adam([{k: v for k, v in gr.items() if k != 'name'} for gr in groups])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
    criterion, class_w = make_criterion(s['condition'], counts)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and device.type == 'cuda')

    best_loss, best_ep, bad, hist = float('inf'), -1, 0, []
    best_path = os.path.join(rdir, 'best.pth'); best_state = None
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    t0 = time.time(); n_ep = 3 if SMOKE_TEST else NUM_EPOCHS
    for ep in range(1, n_ep + 1):
        te0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y in tr_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == 'cuda'):
                out = model(x)
            loss = criterion(out.float(), y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tl += loss.item() * x.size(0); tc += (out.argmax(1) == y).sum().item(); tn += x.size(0)
        train_time = time.time() - te0
        vl = eval_loss(model, va_loader, criterion)
        pv, yv = predict(model, va_loader)
        scheduler.step(vl)
        row = dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn, val_loss=vl, val_acc=float((pv.argmax(1) == yv).mean()),
                   lr_head=optimizer.param_groups[-1]['lr'], epoch_train_sec=train_time, epoch_total_sec=time.time() - te0)
        if LOG_TEST_EVERY_EPOCH:
            pt, yt = predict(model, te_loader); row['test_macro_f1_REPORT_ONLY'] = macro_f1_fast(yt, pt.argmax(1))
        hist.append(row)
        improved = vl < best_loss
        if improved:
            best_loss, best_ep, bad = vl, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        print(f"[{rid}] ep {ep:03d} train {tl/tn:.4f}/{tc/tn:.3f}  val {vl:.4f}/{row['val_acc']:.3f}"
              f"  {row['epoch_total_sec']:.0f}s{'  *' if improved else ''}", flush=True)
        if bad >= PATIENCE: break
    total_time = time.time() - t0
    final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ── evaluate both checkpoints on val and test ──
    rows = []
    for ck, st in [('best', best_state), ('final', final_state)]:
        model.load_state_dict(st)
        for split, loader, items in [('val', va_loader, va), ('test', te_loader, te)]:
            P, Y = predict(model, loader)
            for i, ((pth, y), p) in enumerate(zip(items, P)):
                rows.append([split, ck, i, os.path.relpath(pth, ORIGINAL_ROOT), int(y), *map(float, p)])
    preds = pd.DataFrame(rows, columns=['split', 'ckpt', 'idx', 'relpath', 'y_true'] + [f'p_{c}' for c in CLASS_NAMES])
    preds.to_csv(os.path.join(rdir, 'preds.csv'), index=False)
    pd.DataFrame(hist).to_csv(os.path.join(rdir, 'history.csv'), index=False)
    if SAVE_WEIGHTS in ('best', 'both'): torch.save(best_state, best_path)
    if SAVE_WEIGHTS == 'both': torch.save(final_state, os.path.join(rdir, 'final.pth'))

    summ = {}
    for ck in ['best', 'final']:
        d = preds[(preds.split == 'test') & (preds.ckpt == ck)]
        m = metrics_from_cm(confusion(d.y_true.values, d[[f'p_{c}' for c in CLASS_NAMES]].values.argmax(1)))
        summ[f'test_{ck}'] = {k: float(v) for k, v in m.items()}
    meta = dict(status='done', run_id=rid, **{k: v for k, v in s.items()}, **count_params(model),
                train_counts=counts.tolist(), class_weights=class_w, n_train_samples=len(index),
                epochs_trained=len(hist), best_epoch=best_ep, best_val_loss=best_loss,
                total_train_sec=total_time, mean_epoch_train_sec=float(np.mean([h['epoch_train_sec'] for h in hist])),
                peak_gpu_mem_mb=(torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None),
                gpu=(torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu'), torch=torch.__version__,
                use_amp=USE_AMP, batch_size=BATCH_SIZE, lr_groups=[(gr['name'], gr['lr']) for gr in groups],
                smoke_test=SMOKE_TEST, **summ)
    json.dump(meta, open(os.path.join(rdir, 'metrics.json'), 'w'), indent=1)
    print(f"[{rid}] DONE {total_time/60:.1f} min | best ep {best_ep}/{len(hist)} | test macro-F1 "
          f"best {summ['test_best']['macro_f1']:.4f} final {summ['test_final']['macro_f1']:.4f}")
    del model, optimizer; torch.cuda.empty_cache() if device.type == 'cuda' else None
    return meta

In [ ]:
# ── CONSOLIDATE: copy every finished run found in attached inputs into this version's output ──
# Makes the output of the LATEST version contain ALL runs, so notebooks 03/04/05 need only one input.
# Re-running with an empty queue (all done) is CPU-only: it just consolidates.
import shutil
copied, have_w = 0, 0
for f in glob.glob('/kaggle/input/**/results/*/metrics.json', recursive=True):
    try:
        if json.load(open(f)).get('status') != 'done': continue
    except Exception: continue
    src = os.path.dirname(f); dst = os.path.join(OUT_DIR, os.path.basename(src))
    if not os.path.exists(os.path.join(dst, 'metrics.json')):
        shutil.copytree(src, dst, dirs_exist_ok=True); copied += 1
done_dirs = [d for d in glob.glob(OUT_DIR + '/*') if os.path.exists(d + '/metrics.json')]
have_w = sum(os.path.exists(d + '/best.pth') for d in done_dirs)
missing_preds = [os.path.basename(d) for d in done_dirs if not os.path.exists(d + '/preds.csv')]
print(f'copied {copied} runs from inputs | {len(done_dirs)} finished runs now in {OUT_DIR} | with best.pth: {have_w}')
if missing_preds: print('WARNING: runs without preds.csv:', missing_preds)
print('still missing from QUEUE:', [run_id(s) for s in QUEUE if not os.path.exists(os.path.join(OUT_DIR, run_id(s), 'metrics.json'))])

In [ ]:
# ── EXECUTE QUEUE ─────────────────────────────────────────────────────────────
session_t0 = time.time(); done = finished_ids(); durations = []
todo = [s for s in QUEUE if run_id(s) not in done]
print(f'{len(QUEUE)} queued, {len(QUEUE) - len(todo)} already finished, {len(todo)} to run')
for s in todo:
    elapsed_h = (time.time() - session_t0) / 3600
    est_h = (np.median(durations) / 3600) if durations else 1.0
    if elapsed_h + est_h > MAX_SESSION_HOURS:
        print(f'Stopping: {elapsed_h:.1f} h used; next run (~{est_h:.1f} h) would exceed the session limit.'); break
    if s['opts'].get('lungcrop') and not any(glob.glob(g, recursive=True) for g in LUNG_BOX_GLOBS):
        print(f'skipping {run_id(s)}: lung_boxes.csv not attached (run notebook 07 first)'); continue
    t = time.time(); train_one(s); durations.append(time.time() - t)

In [ ]:
# ── BUDGET PLANNER: measured cost so far → remaining queue estimate ───────────
recs = []
for g in RESUME_GLOBS:
    for f in glob.glob(g, recursive=True):
        try: recs.append(json.load(open(f)))
        except Exception: pass
if recs:
    R = pd.DataFrame(recs).drop_duplicates('run_id')
    per = R.groupby('arch').total_train_sec.median() / 3600
    print('Median GPU-hours per run by architecture (measured):\n', per.round(2))
    left = [s for s in QUEUE if run_id(s) not in set(R.run_id)]
    est = sum(per.get(s['arch'], per.mean()) for s in left)
    print(f'{len(left)} runs left ≈ {est:.1f} GPU-hours')